# dplyr / tidyr で学ぶ R データ前処理 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** の R カーネル（xeus-r）で、
tidyverse の中心パッケージ **dplyr**（データ操作）と **tidyr**（データ整形）を使った
データ前処理の基本を学ぶチュートリアルです。

## 対象者
- R の基本文法（`r/r_beginner_tutorial.ipynb`）を終えた方
- Excel でやっていた「絞り込み・集計・ピボット」を R で再現したい方
- データ分析の前段となる「データの掃除」を体系的に学びたい方

## このチュートリアルで学ぶこと
0. 準備（パッケージの読み込みと注意点）
1. tibble — モダンなデータフレーム
2. 行と列の選択（select / filter / arrange）
3. 新しい列を作る（mutate / if_else / case_when）
4. グループ集計（group_by / summarise / count / across）
5. 表の結合（left_join / inner_join / anti_join）
6. 縦持ち・横持ち変換（pivot_longer / pivot_wider）
7. 列の分割と連結（separate / unite）
8. 欠損値の処理（drop_na / replace_na / coalesce）
9. 集計結果の可視化（base graphics）
10. 総合演習 — 売上データの前処理パイプライン

各章の最後には練習問題があります。まず自分で書いてみて、
「解答例を見る」をクリックして答え合わせをしてください。

---
## 0. 準備 — パッケージの読み込みと注意点

このサイトの R 環境（xeus-r）には、サイトのビルド時に **dplyr・tidyr・tibble が組み込み済み** です。
`library()` で読み込むだけで使えます（`install.packages()` はブラウザ内では使えません）。

覚えておきたい注意点が 2 つあります。

1. **読み込み時のメッセージ**：dplyr を読み込むと「`filter` などが base R の関数を隠しました」という
   メッセージが出ます。これは正常な動作ですが、このノートでは
   `suppressPackageStartupMessages()` で表示を抑えています。
2. **関数名の衝突**：dplyr 読み込み後、`filter()` は dplyr のもの（行の絞り込み）が優先されます。
   元の `stats::filter()`（移動平均）を使いたいときは `stats::filter()` と書きます。

また、パイプ演算子はこのノートでは R 4.1 から標準搭載の **ネイティブパイプ `|>`** を使います。
`x |> f(y)` は `f(x, y)` と同じ意味で、「データを左から右へ流す」ように処理を書けます。

In [ ]:
# タイムゾーンとグラフの設定（JupyterLite 向けのおまじない）
Sys.setenv(TZ = "Asia/Tokyo")
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 100, jupyter.plot_scale = 1)

# パッケージの読み込み（起動時メッセージは抑える）
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(tibble)
})

cat("R     :", R.version.string, "\n")
cat("dplyr :", as.character(packageVersion("dplyr")), "\n")
cat("tidyr :", as.character(packageVersion("tidyr")), "\n")
cat("tibble:", as.character(packageVersion("tibble")), "\n")

---
## 1. tibble — モダンなデータフレーム

**tibble** は data.frame の改良版です。中身は data.frame と同じように使えますが、
表示が見やすく、分析の途中で扱いやすい工夫がされています。

| 項目 | data.frame | tibble |
|---|---|---|
| 表示 | 全行・全列を出力 | 先頭 10 行と画面に収まる列だけ。**列の型も表示** |
| 部分列 `df[, 1]` | ベクトルに変わることがある | 常に tibble のまま |
| 文字列 | 昔は自動で因子化されていた | そのまま文字列 |

作るには `tibble(列名 = 値, ...)` を使います。既存の data.frame は `as_tibble()` で変換できます。
`glimpse()` を使うと「列名・型・最初の値」を横向きに一覧できます。

In [ ]:
# 商品マスタを tibble で作る
products <- tibble(
  product_id = c("P-01", "P-02", "P-03"),
  name       = c("Coffee", "Tea", "Juice"),
  category   = c("Hot", "Hot", "Cold"),
  price      = c(450, 400, 380)
)
products

In [ ]:
# tibble は「行数×列数」と各列の型（<chr> や <dbl>）を一緒に表示してくれる
glimpse(products)

# data.frame との違い：1 列だけ取り出したとき
df <- as.data.frame(products)
print(class(df[, "price"]))          # data.frame はベクトルになる
class(products[, "price"])           # tibble は tibble のまま（列を保つ）

### 練習問題 1

次の 3 店舗の情報を `stores` という tibble で作り、`glimpse()` で構造を確認してから、
`nrow()` と `ncol()` で行数・列数を表示してください。

| store | city | opened |
|---|---|---|
| S-1 | Tokyo | 2018 |
| S-2 | Osaka | 2020 |
| S-3 | Nagoya | 2023 |

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```r
stores <- tibble(
  store  = c("S-1", "S-2", "S-3"),
  city   = c("Tokyo", "Osaka", "Nagoya"),
  opened = c(2018, 2020, 2023)
)
glimpse(stores)
cat("行数:", nrow(stores), " 列数:", ncol(stores), "\n")
```

</details>

---
## 2. 行と列の選択 — select / filter / arrange

dplyr の基本動詞はどれも「**tibble を受け取り、tibble を返す**」関数です。
そのためパイプ `|>` でいくらでもつなげられます。

| 関数 | 役割 | 例 |
|---|---|---|
| `select()` | **列**を選ぶ | `select(store, qty)`、`select(-month)`、`select(starts_with("p"))` |
| `filter()` | **行**を条件で絞り込む | `filter(qty >= 20, store == "Tokyo")` |
| `arrange()` | 行を並べ替える | `arrange(desc(qty))` |

まず、このノート全体で使う売上データを作ります。`expand_grid()`（tidyr）は
「すべての組み合わせ」の表を作る関数です。

In [ ]:
# 3 店舗 × 3 か月 × 3 商品 = 27 行の売上データを作る
set.seed(2026)  # 乱数を固定して、実行のたびに同じデータになるようにする
sales <- expand_grid(
  store      = c("Tokyo", "Osaka", "Nagoya"),
  month      = c("2026-01", "2026-02", "2026-03"),
  product_id = c("P-01", "P-02", "P-03")
) |>
  mutate(qty = sample(5:30, n(), replace = TRUE))   # 販売数量（5〜30 の乱数）

sales

In [ ]:
# select：列を選ぶ・外す・パターンで選ぶ
print(sales |> select(store, qty))            # 名前で選ぶ
print(sales |> select(-month))                # month 列を外す
sales |> select(starts_with("p"))             # p で始まる列（product_id）

In [ ]:
# filter：条件に合う行だけ残す
print(sales |> filter(store == "Tokyo"))                  # 東京だけ
print(sales |> filter(qty >= 25))                         # 25 個以上売れた行
sales |> filter(store %in% c("Tokyo", "Osaka"), qty >= 25) # 複数条件は「,」で AND

In [ ]:
# arrange：並べ替え（desc() で降順）
print(sales |> arrange(desc(qty)) |> head(5))   # 売れた順に上位 5 行

# slice_max()：上位 n 行を直接取り出す
sales |> slice_max(qty, n = 3)

### 練習問題 2

`sales` から次の表を作ってください。

1. 大阪（Osaka）の 2026-02 の行だけに絞り込む
2. 列は `product_id` と `qty` だけにする
3. `qty` の降順に並べる

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```r
sales |>
  filter(store == "Osaka", month == "2026-02") |>
  select(product_id, qty) |>
  arrange(desc(qty))
```

</details>

---
## 3. 新しい列を作る — mutate / if_else / case_when

`mutate()` は既存の列を使って新しい列を作ります（既存の列名を指定すれば上書き）。
条件によって値を変えたいときは次の 2 つを使います。

- `if_else(条件, 真のとき, 偽のとき)` — 2 分岐
- `case_when(条件1 ~ 値1, 条件2 ~ 値2, ..., TRUE ~ それ以外)` — 多分岐（上から順に判定）

そのほか、列名の変更は `rename(新しい名前 = 古い名前)`、
列の並び替えは `relocate(列, .before = 列)` を使います。

In [ ]:
# mutate で列を追加する
sales2 <- sales |>
  mutate(
    size  = if_else(qty >= 20, "large", "small"),
    grade = case_when(
      qty >= 25 ~ "A",
      qty >= 15 ~ "B",
      TRUE      ~ "C"      # どの条件にも当てはまらないとき
    )
  )
sales2 |> head(8)

In [ ]:
# rename と relocate
sales2 |>
  rename(quantity = qty) |>          # qty 列の名前を quantity に変える
  relocate(month, .before = store) |> # month 列を store の前に移動
  head(3)

### 練習問題 3

`sales` に次の 2 列を追加して、先頭 6 行を表示してください。

1. `is_hot`：`product_id` が "P-01" か "P-02"（ホット飲料）なら `TRUE`、そうでなければ `FALSE`
   （ヒント：`%in%` が使えます）
2. `demand`：`qty` が 25 以上なら "high"、15 以上なら "mid"、それ以外は "low"

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```r
sales |>
  mutate(
    is_hot = product_id %in% c("P-01", "P-02"),
    demand = case_when(
      qty >= 25 ~ "high",
      qty >= 15 ~ "mid",
      TRUE      ~ "low"
    )
  ) |>
  head(6)
```

</details>

---
## 4. グループ集計 — group_by / summarise

「店舗ごとの合計」「月ごとの平均」のような集計は、
**`group_by()` でグループを宣言 → `summarise()` で 1 グループ 1 行に要約** という流れで書きます。

- `n()` はグループの行数（summarise の中だけで使える）
- `summarise()` には **`.groups = "drop"`** を付けて、集計後にグループ状態を解除するのがおすすめです
  （付けないと「まだグループが残っている」という通知が表示されることがあります）
- 単に行数を数えるだけなら `count()` が近道です
- 複数の列に同じ集計を適用するときは `across()` を使います

In [ ]:
# 店舗ごとの合計・平均・行数
sales |>
  group_by(store) |>
  summarise(
    total = sum(qty),
    avg   = mean(qty),
    rows  = n(),
    .groups = "drop"
  )

In [ ]:
# 2 つの列でグループ化（店舗 × 月）
monthly <- sales |>
  group_by(store, month) |>
  summarise(total = sum(qty), .groups = "drop")
print(monthly)

# count()：組み合わせの行数を数える（sort = TRUE で多い順）
sales |> count(store, month, sort = TRUE) |> head(3)

In [ ]:
# across()：複数の列にまとめて集計関数を適用する
sales |>
  mutate(revenue = qty * 100) |>    # 仮の売上額の列を作って
  group_by(store) |>
  summarise(across(c(qty, revenue), list(total = sum, mean = mean)), .groups = "drop")

### 練習問題 4

`sales` を使って、**商品ごと（product_id ごと）** に

- 合計販売数 `total`
- 最大販売数 `max_qty`

を集計し、`total` の降順で表示してください（`.groups = "drop"` を忘れずに）。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```r
sales |>
  group_by(product_id) |>
  summarise(
    total   = sum(qty),
    max_qty = max(qty),
    .groups = "drop"
  ) |>
  arrange(desc(total))
```

</details>

---
## 5. 表の結合 — left_join / inner_join / anti_join

売上データには商品 ID しかなく、商品名や単価は別の「マスタ表」にある——
実務のデータはたいていこの形です。2 つの表は **キー列** でつなぎます。

| 関数 | 残る行 | 用途 |
|---|---|---|
| `left_join(x, y, by = "キー")` | x の全行（y に無ければ NA） | マスタ情報の付与 |
| `inner_join(x, y, by = "キー")` | 両方にある行だけ | 突き合わせ |
| `anti_join(x, y, by = "キー")` | y に **無い** x の行 | 漏れ・不一致の検出 |

`by =` は省略できますが、**明示するのが安全** です（意図しない列で結合される事故を防げます）。

In [ ]:
# 商品マスタ（1 章で作った products）を売上に結合して、売上額を計算する
sales_full <- sales |>
  left_join(products, by = "product_id") |>
  mutate(revenue = qty * price)
sales_full |> head(6)

In [ ]:
# anti_join：キャンペーン対象になっていない商品を洗い出す
campaign <- tibble(product_id = c("P-01", "P-03"), discount = c(0.1, 0.2))

print(sales |> inner_join(campaign, by = "product_id") |> head(4))   # 対象商品の売上だけ
products |> anti_join(campaign, by = "product_id")                   # 対象外の商品

### 練習問題 5

`sales_full` を使って、**カテゴリ（category）× 店舗（store）** ごとの売上額合計 `total_rev` を集計し、
`total_rev` の降順で表示してください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```r
sales_full |>
  group_by(category, store) |>
  summarise(total_rev = sum(revenue), .groups = "drop") |>
  arrange(desc(total_rev))
```

</details>

---
## 6. 縦持ち・横持ち変換 — pivot_longer / pivot_wider

- **縦持ち（long）**：1 行 = 1 観測。集計やグラフに向く形
- **横持ち（wide）**：行 × 列のクロス表。人が読むのに向く形

tidyr の `pivot_wider()`（縦 → 横）と `pivot_longer()`（横 → 縦）で相互に変換できます。

In [ ]:
# 縦持ちの集計表（店舗 × 月）を、月を列に持つクロス表にする
wide <- monthly |>
  pivot_wider(names_from = month, values_from = total)
print(wide)

# クロス表を縦持ちに戻す
wide |>
  pivot_longer(cols = -store, names_to = "month", values_to = "total") |>
  head(5)

### 練習問題 6

`monthly`（店舗 × 月の合計）を、今度は **店舗を列** に持つクロス表
（行 = month、列 = Tokyo / Osaka / Nagoya）に変換してください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```r
monthly |>
  pivot_wider(names_from = store, values_from = total)
```

</details>

---
## 7. 列の分割と連結 — separate / unite

"2026-01" のように **1 つの列に複数の情報が入っている** ときは `separate()` で分割し、
逆に複数の列を 1 つにまとめるときは `unite()` を使います。

- `separate(列, into = c("新列1", "新列2"), sep = "-")`
- `unite(新列, 列1, 列2, sep = "-")`

In [ ]:
# month 列（"2026-01"）を year と mon に分ける
sales_ym <- sales |>
  separate(month, into = c("year", "mon"), sep = "-")
print(sales_ym |> head(4))

# 分けた列を元に戻す
sales_ym |>
  unite(month, year, mon, sep = "-") |>
  head(4)

### 練習問題 7

`products` の `product_id`（"P-01" など）を `sep = "-"` で `prefix` と `num` の 2 列に分割し、
結果を表示してください。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```r
products |>
  separate(product_id, into = c("prefix", "num"), sep = "-")
```

</details>

---
## 8. 欠損値の処理 — drop_na / replace_na / coalesce

実データには欠損値 `NA` が付きものです。方針は大きく 2 つ：

- **除外する**：`drop_na()`（NA を含む行を落とす。`drop_na(列)` で特定の列だけ見る）
- **埋める**：`replace_na(list(列 = 値))`、または `coalesce(優先する列, 予備の列, 既定値)`

その前に、まず **どの列にいくつ NA があるか** を数えて全体像をつかみましょう。

In [ ]:
# 欠損を含むアンケートデータ
survey <- tibble(
  id    = 1:6,
  score = c(80, NA, 65, 90, NA, 72),
  email = c("a@x.jp", NA, "c@x.jp", NA, "e@x.jp", "f@x.jp"),
  phone = c(NA, "090-1111", NA, "090-2222", "090-3333", NA)
)
print(survey)

# 列ごとの NA の個数
survey |> summarise(across(everything(), ~ sum(is.na(.x))))

In [ ]:
# 1) score が NA の行を除外する
print(survey |> drop_na(score))

# 2) score の NA を 0 で埋める
print(survey |> replace_na(list(score = 0)))

# 3) 連絡先：email があれば email、無ければ phone、両方無ければ "none"
survey |>
  mutate(contact = coalesce(email, phone, "none")) |>
  select(id, contact)

### 練習問題 8

`survey` について次の 2 つを求めてください。

1. `drop_na()`（全列対象）で NA を 1 つでも含む行を落とすと、何行残りますか（`nrow()` で表示）
2. `score` の NA を **NA を除いた平均値** で埋めた列 `score_filled` を作り、`id` と `score_filled` を表示
   （ヒント：`mean(score, na.rm = TRUE)`）

In [ ]:
# 練習問題 8 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 8 の解答例を見る</strong></summary>

```r
# 1. 全列に NA が無い行だけ残す
cat("残る行数:", nrow(survey |> drop_na()), "\n")

# 2. 平均値で埋める
survey |>
  mutate(score_filled = replace_na(score, mean(score, na.rm = TRUE))) |>
  select(id, score_filled)
```

</details>

---
## 9. 集計結果の可視化（base graphics）

集計した tibble は、そのまま base R のグラフ関数に渡せます。
この環境では **グラフ内の文字に日本語を使うと文字化けする**（豆腐 □ になる）ため、
タイトルや軸ラベルは英語で書きます（Markdown の説明は日本語で OK）。

In [ ]:
# 店舗ごとの合計販売数を棒グラフに
by_store <- sales |>
  group_by(store) |>
  summarise(total = sum(qty), .groups = "drop")
print(by_store)

barplot(by_store$total, names.arg = by_store$store,
        main = "Total quantity by store", ylab = "Quantity",
        col = "steelblue")

In [ ]:
# 店舗ごとの月次推移を折れ線で重ねる（pivot_wider で行列にしてから matplot）
trend <- monthly |>
  pivot_wider(names_from = store, values_from = total)
print(trend)

matplot(x = 1:3, y = as.matrix(trend[, -1]), type = "b", pch = 16, lty = 1,
        col = c("steelblue", "tomato", "seagreen"),
        xlab = "Month (1 = Jan 2026)", ylab = "Quantity",
        main = "Monthly quantity by store", xaxt = "n")
axis(1, at = 1:3, labels = c("Jan", "Feb", "Mar"))
legend("topleft", legend = colnames(trend)[-1], pch = 16, lty = 1,
       col = c("steelblue", "tomato", "seagreen"))

### 練習問題 9

`sales_full` を使って **カテゴリ（category）ごとの売上額合計** を集計し、
棒グラフにしてください（タイトル・ラベルは英語で。例："Revenue by category"）。

In [ ]:
# 練習問題 9 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 9 の解答例を見る</strong></summary>

```r
by_cat <- sales_full |>
  group_by(category) |>
  summarise(total_rev = sum(revenue), .groups = "drop")
print(by_cat)

barplot(by_cat$total_rev, names.arg = by_cat$category,
        main = "Revenue by category", ylab = "Revenue (JPY)",
        col = c("tomato", "steelblue"))
```

</details>

---
## まとめ

| 章 | 学んだこと |
|---|---|
| tibble | `tibble()`、`as_tibble()`、`glimpse()` |
| 選択 | `select()`（列）、`filter()`（行）、`arrange()` / `slice_max()`（並べ替え） |
| 列の作成 | `mutate()`、`if_else()`、`case_when()`、`rename()`、`relocate()` |
| 集計 | `group_by()` + `summarise(.groups = "drop")`、`n()`、`count()`、`across()` |
| 結合 | `left_join()` / `inner_join()` / `anti_join()`（`by =` は明示する） |
| 整形 | `pivot_wider()` / `pivot_longer()`、`separate()` / `unite()` |
| 欠損 | `drop_na()`、`replace_na()`、`coalesce()` |

**書き方のコツ**：処理はパイプ `|>` で「データ → 動詞 → 動詞 → …」の順につなげる。
1 ステップずつ実行して途中結果を確認しながら組み立てると、間違いにすぐ気づけます。

## 次のステップ
- `r/r_beginner_tutorial.ipynb` — R の基本文法（復習）
- `r/r_timeseries_beginner_tutorial.ipynb` — 時系列分析入門（base R）
- `jupyterlite/jupyterlite_xeus_r_stats_practice.ipynb` — 統計テスト演習

---
## 総合演習 — 売上データの前処理パイプライン

現場から届いた「汚れた」売上データを、この章までの道具で分析できる形に整えましょう。

**課題**：次のセルで作られる `raw`（欠損あり・商品名なし）と `master` を使って、

1. `qty` が `NA` の行の数を数え、`NA` を 0 で埋める
2. `master` を `left_join` して `revenue = qty * price` を計算する
3. `month` を `year` と `mon` に分割する
4. 店舗 × カテゴリ別の売上額合計を求め、カテゴリを列に持つクロス表にする
5. 店舗ごとの売上額合計を棒グラフにする（英語ラベル）

まず自分でパイプラインを書いてから、解答例と見比べてください。

In [ ]:
# 汚れた売上データ（このセルを実行してから取り組んでください）
set.seed(7)
raw <- expand_grid(
  store      = c("Tokyo", "Osaka", "Nagoya"),
  month      = c("2026-01", "2026-02", "2026-03"),
  product_id = c("P-01", "P-02", "P-03")
) |>
  mutate(qty = sample(c(NA, 5:30), n(), replace = TRUE))   # ところどころ NA

master <- tibble(
  product_id = c("P-01", "P-02", "P-03"),
  name       = c("Coffee", "Tea", "Juice"),
  category   = c("Hot", "Hot", "Cold"),
  price      = c(450, 400, 380)
)

cat("raw:", nrow(raw), "行\n")
raw |> head(5)

<details>
<summary><strong>総合演習の解答例を見る</strong></summary>

```r
# 1. 欠損の確認と補完
cat("qty が NA の行数:", sum(is.na(raw$qty)), "\n")
clean <- raw |> replace_na(list(qty = 0))

# 2. マスタを結合して売上額を計算
clean <- clean |>
  left_join(master, by = "product_id") |>
  mutate(revenue = qty * price)
print(head(clean, 5))

# 3. month を year / mon に分割
clean <- clean |> separate(month, into = c("year", "mon"), sep = "-")

# 4. 店舗 × カテゴリのクロス表
cross <- clean |>
  group_by(store, category) |>
  summarise(total_rev = sum(revenue), .groups = "drop") |>
  pivot_wider(names_from = category, values_from = total_rev)
print(cross)

# 5. 店舗ごとの売上額合計を棒グラフに
by_store2 <- clean |>
  group_by(store) |>
  summarise(total_rev = sum(revenue), .groups = "drop")
barplot(by_store2$total_rev, names.arg = by_store2$store,
        main = "Total revenue by store", ylab = "Revenue (JPY)", col = "steelblue")
```

</details>

お疲れさまでした！ `filter` → `mutate` → `join` → `group_by` → `pivot` という流れは、
どんなデータでも繰り返し登場する「前処理の型」です。
自分のデータでも、まず tibble にしてこの型に当てはめてみてください。